# Imports

In [1]:
import optuna
import pickle

import numpy as np
import pandas as pd

from utils import load_pickle
from lightgbm import LGBMClassifier

from sklearn.metrics import log_loss, balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

/home/junior/Documentos/GitHub/kaggle-competition-predicting-stellar-class/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
label_encoder = load_pickle('../models/label_encoder.pkl')

# Loading Datasets

In [3]:
X_train = pd.read_parquet('../data/X_train_stacking_layer_three.parquet')
y_train = pd.read_parquet('../data/y_train.parquet')

X_test = pd.read_parquet('../data/X_test_stacking_layer_three.parquet')

In [4]:
X_train.head()

,lgbm_0,lgbm_1,lgbm_2,cat_0,cat_1,cat_2,xgb_0,xgb_1,xgb_2,hist_0,hist_1,hist_2,rf_0,rf_1,rf_2,extra_0,extra_1,extra_2
0,0.999778,0.000212,0.000009,0.999787,0.000174,0.000040,0.999864,0.000121,0.000016,0.999733,0.000264,2.459617e-06,0.999775,0.000225,2.616556e-08,0.998912,0.000955,0.000133
1,0.994627,0.000176,0.005197,0.995247,0.000365,0.004388,0.992235,0.000199,0.007566,0.973059,0.000137,2.680453e-02,0.994737,0.000047,5.216709e-03,0.976808,0.000850,0.022342
2,0.000048,0.999944,0.000008,0.000483,0.999485,0.000032,0.000169,0.999822,0.000009,0.000015,0.999984,1.319155e-06,0.000011,0.999989,9.836632e-08,0.000068,0.999897,0.000035
3,0.999915,0.000081,0.000005,0.999745,0.000217,0.000039,0.999858,0.000127,0.000014,0.999960,0.000040,6.602657e-07,0.999793,0.000207,3.828407e-08,0.998919,0.000947,0.000134
4,0.998164,0.001812,0.000024,0.998143,0.001764,0.000093,0.998504,0.001459,0.000037,0.986426,0.013556,1.806930e-05,0.998227,0.001769,3.566911e-06,0.994430,0.005248,0.000322


In [5]:
X_test.head()

,lgbm_0,lgbm_1,lgbm_2,cat_0,cat_1,cat_2,xgb_0,xgb_1,xgb_2,hist_0,hist_1,hist_2,rf_0,rf_1,rf_2,extra_0,extra_1,extra_2
0,0.998177,0.001784,0.000039,0.997654,0.002153,0.000193,0.998108,0.001843,0.000049,0.997980,0.002008,0.000012,0.998183,0.001800,0.000017,0.993598,0.005832,0.000569
1,0.997269,0.002710,0.000021,0.997945,0.001978,0.000077,0.997185,0.002778,0.000037,0.994660,0.005324,0.000015,0.997905,0.002092,0.000003,0.993669,0.005970,0.000361
2,0.996089,0.000609,0.003303,0.997698,0.000711,0.001592,0.997502,0.000382,0.002115,0.997295,0.001499,0.001207,0.997506,0.000715,0.001780,0.988872,0.002584,0.008544
3,0.000643,0.000107,0.999250,0.001658,0.000165,0.998177,0.000730,0.000132,0.999138,0.000139,0.000020,0.999841,0.000541,0.000023,0.999436,0.000378,0.000214,0.999409
4,0.999631,0.000357,0.000012,0.999578,0.000365,0.000058,0.999737,0.000245,0.000018,0.999526,0.000469,0.000005,0.999785,0.000205,0.000010,0.998670,0.001116,0.000214


# Machine Learning

In [6]:
def objective(trial, X, y):

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scores = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train_fold = X.iloc[train_idx, :]
        X_valid_fold = X.iloc[valid_idx, :]

        y_train_fold = y.iloc[train_idx]
        y_valid_fold = y.iloc[valid_idx]

        model = model = LGBMClassifier(
            objective='multiclass',
            metric='multi_logloss',
            boosting_type='gbdt',
            verbosity=-1,
            random_state=42,
            n_jobs=1,
            n_estimators=trial.suggest_int('n_estimators', 50, 500),
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            num_leaves=trial.suggest_int('num_leaves', 2, 32),
            max_depth=trial.suggest_int('max_depth', 1, 6),
            min_child_samples=trial.suggest_int('min_child_samples', 5, 100),
            lambda_l1=trial.suggest_float('lambda_l1', 1e-5, 10.0, log=True),
            lambda_l2=trial.suggest_float('lambda_l2', 1e-5, 10.0, log=True),
            feature_fraction=trial.suggest_float('feature_fraction', 0.7, 1.0),
            bagging_fraction=trial.suggest_float('bagging_fraction', 0.7, 1.0),
            bagging_freq=trial.suggest_int('bagging_freq', 1, 5),
        ).fit(X_train_fold, y_train_fold)

        proba = model.predict_proba(X_valid_fold)

        w0 = trial.suggest_float('weight_class_0', 0.1, 2.0)
        w1 = trial.suggest_float('weight_class_1', 0.1, 2.0)
        w2 = trial.suggest_float('weight_class_2', 0.1, 2.0)

        weighted_probas = proba * np.array([w0, w1, w2])

        pred = np.argmax(weighted_probas, axis=1)
        
        score = balanced_accuracy_score(y_valid_fold, pred)
        scores.append(score)

        trial.report(np.mean(scores), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(scores)


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train, y_train.class_encoded), n_trials=60, n_jobs=-1, show_progress_bar=True)


print("Best trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-06-18 15:05:46,241] A new study created in memory with name: no-name-405e4455-e2a5-42d9-a92d-8ea75a3ab31c
Best trial: 1. Best value: 0.951342:   2%|██▎                                                                                                                                        | 1/60 [00:36<35:52, 36.48s/it]

[I 2026-06-18 15:06:22,722] Trial 1 finished with value: 0.9513424390358345 and parameters: {'n_estimators': 68, 'learning_rate': 0.2112075956127585, 'num_leaves': 2, 'max_depth': 3, 'min_child_samples': 99, 'lambda_l1': 0.010096595800012756, 'lambda_l2': 0.007033843714582129, 'feature_fraction': 0.7809207419147921, 'bagging_fraction': 0.7936875031290453, 'bagging_freq': 1, 'weight_class_0': 1.945284331981904, 'weight_class_1': 1.6598249293411205, 'weight_class_2': 1.0335848111484525}. Best is trial 1 with value: 0.9513424390358345.


Best trial: 1. Best value: 0.951342:   3%|████▋                                                                                                                                      | 2/60 [01:20<39:47, 41.17s/it]

[I 2026-06-18 15:07:07,173] Trial 5 finished with value: 0.9123742554601068 and parameters: {'n_estimators': 112, 'learning_rate': 0.015273323381813753, 'num_leaves': 9, 'max_depth': 3, 'min_child_samples': 41, 'lambda_l1': 0.08837170918538327, 'lambda_l2': 0.15954364044901326, 'feature_fraction': 0.9978902474865221, 'bagging_fraction': 0.7735702233514751, 'bagging_freq': 4, 'weight_class_0': 1.4453039703489543, 'weight_class_1': 1.1110714797560481, 'weight_class_2': 0.2628112957828161}. Best is trial 1 with value: 0.9513424390358345.


Best trial: 8. Best value: 0.960259:   5%|██████▉                                                                                                                                    | 3/60 [01:37<28:17, 29.78s/it]

...

[I 2026-06-18 15:24:38,932] Trial 55 finished with value: 0.9639316501763598 and parameters: {'n_estimators': 422, 'learning_rate': 0.09622801708419597, 'num_leaves': 21, 'max_depth': 1, 'min_child_samples': 80, 'lambda_l1': 0.0009801494909942032, 'lambda_l2': 0.00022483973641609098, 'feature_fraction': 0.7522698014557753, 'bagging_fraction': 0.8678331838879346, 'bagging_freq': 2, 'weight_class_0': 0.49309835487558873, 'weight_class_1': 0.5117346162129203, 'weight_class_2': 1.5236044920805272}. Best is trial 56 with value: 0.9660721332121811.


Best trial: 56. Best value: 0.966072:  98%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋  | 59/60 [19:57<00:26, 26.09s/it]

[I 2026-06-18 15:25:43,864] Trial 58 finished with value: 0.9633612022017264 and parameters: {'n_estimators': 320, 'learning_rate': 0.10723274321932065, 'num_leaves': 18, 'max_depth': 5, 'min_child_samples': 79, 'lambda_l1': 0.00011589054763669146, 'lambda_l2': 2.5402965093386674, 'feature_fraction': 0.7558500747498834, 'bagging_fraction': 0.8842133465299639, 'bagging_freq': 1, 'weight_class_0': 0.47801378645144676, 'weight_class_1': 1.5584946378805118, 'weight_class_2': 0.9456920553897927}. Best is trial 56 with value: 0.9660721332121811.


Best trial: 56. Best value: 0.966072: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 60/60 [19:59<00:00, 19.99s/it]

[I 2026-06-18 15:25:45,839] Trial 59 finished with value: 0.966059648476878 and parameters: {'n_estimators': 320, 'learning_rate': 0.017995893512805426, 'num_leaves': 18, 'max_depth': 5, 'min_child_samples': 81, 'lambda_l1': 0.0001203526717084888, 'lambda_l2': 2.2329434297224835, 'feature_fraction': 0.7567590223750145, 'bagging_fraction': 0.8873488868931302, 'bagging_freq': 1, 'weight_class_0': 0.5399354599403758, 'weight_class_1': 1.5889596096599146, 'weight_class_2': 1.7708332247091554}. Best is trial 56 with value: 0.9660721332121811.
Best trial score:
0.9660721332121811

Best params:
{'n_estimators': 413, 'learning_rate': 0.10710489288644891, 'num_leaves': 22, 'max_depth': 1, 'min_child_samples': 95, 'lambda_l1': 0.0009444444419897858, 'lambda_l2': 2.624097576059962, 'feature_fraction': 0.7542572256209854, 'bagging_fraction': 0.8617937173210679, 'bagging_freq': 2, 'weight_class_0': 0.25638415553594995, 'weight_class_1': 0.5558775959723516, 'weight_class_2': 1.5326100445071382}


In [18]:
study.optimize(lambda trial: objective(trial, X_train, y_train.class_encoded), n_trials=30, n_jobs=-1, show_progress_bar=True)

Best trial: 56. Best value: 0.966072:   3%|████▌                                                                                                                                  | 1/30 [02:11<1:03:41, 131.79s/it]

[I 2026-06-18 15:31:11,035] Trial 61 finished with value: 0.9658128185813194 and parameters: {'n_estimators': 202, 'learning_rate': 0.019349873436356732, 'num_leaves': 26, 'max_depth': 1, 'min_child_samples': 86, 'lambda_l1': 0.0034247402388475263, 'lambda_l2': 4.67961054609198, 'feature_fraction': 0.7840703383098341, 'bagging_fraction': 0.8966498596469881, 'bagging_freq': 1, 'weight_class_0': 0.26564807318508576, 'weight_class_1': 1.7121371367661486, 'weight_class_2': 1.6896154296926642}. Best is trial 56 with value: 0.9660721332121811.


Best trial: 56. Best value: 0.966072:   7%|█████████▏                                                                                                                                | 2/30 [02:13<25:41, 55.05s/it]

[I 2026-06-18 15:31:12,369] Trial 70 finished with value: 0.9650177600204923 and parameters: {'n_estimators': 205, 'learning_rate': 0.017528748961664065, 'num_leaves': 26, 'max_depth': 1, 'min_child_samples': 86, 'lambda_l1': 0.0009791842774026212, 'lambda_l2': 4.253523578830545, 'feature_fraction': 0.7903085021277989, 'bagging_fraction': 0.856306831310187, 'bagging_freq': 1, 'weight_class_0': 0.2279900133084261, 'weight_class_1': 1.751603307713158, 'weight_class_2': 1.6922084593285576}. Best is trial 56 with value: 0.9660721332121811.


Best trial: 56. Best value: 0.966072:  10%|█████████████▊                                                                                                                            | 3/30 [02:14<13:41, 30.42s/it]

...

[I 2026-06-18 15:36:48,498] Trial 88 pruned. 


Best trial: 56. Best value: 0.966072:  97%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍    | 29/30 [08:51<00:31, 31.25s/it]

[I 2026-06-18 15:37:50,547] Trial 87 finished with value: 0.9656569248277039 and parameters: {'n_estimators': 307, 'learning_rate': 0.03336556663915791, 'num_leaves': 23, 'max_depth': 5, 'min_child_samples': 99, 'lambda_l1': 0.0018899393503089127, 'lambda_l2': 6.757634330733043, 'feature_fraction': 0.7670337732396626, 'bagging_fraction': 0.8862055912633388, 'bagging_freq': 1, 'weight_class_0': 0.5284334956457066, 'weight_class_1': 1.2806818467814254, 'weight_class_2': 1.4633632172238862}. Best is trial 56 with value: 0.9660721332121811.


Best trial: 89. Best value: 0.966166: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 30/30 [09:00<00:00, 18.00s/it]

[I 2026-06-18 15:37:59,413] Trial 89 finished with value: 0.9661655016712585 and parameters: {'n_estimators': 328, 'learning_rate': 0.03392891219249922, 'num_leaves': 20, 'max_depth': 5, 'min_child_samples': 99, 'lambda_l1': 0.0017874228948738, 'lambda_l2': 0.3693642730472445, 'feature_fraction': 0.7650996115316755, 'bagging_fraction': 0.9175761063210657, 'bagging_freq': 1, 'weight_class_0': 0.5111464241626329, 'weight_class_1': 1.819479559256786, 'weight_class_2': 1.8620126135307478}. Best is trial 89 with value: 0.9661655016712585.


In [13]:
lgbm_params = {k: v for k, v in study.best_params.items() if k not in ['weight_class_0', 'weight_class_1', 'weight_class_2']}

lgbm = LGBMClassifier(
    **lgbm_params,
    objective='multiclass',
    metric='multi_logloss',
    boosting_type='gbdt',
    verbosity=-1,
    random_state=42,
    n_jobs=1,
).fit(X_train, y_train.class_encoded)

test_proba = lgbm.predict_proba(X_test)

weights = np.array([study.best_params['weight_class_0'], study.best_params['weight_class_1'], study.best_params['weight_class_2']])
weighted_probas = test_proba * weights

pred = np.argmax(weighted_probas, axis=1)

In [14]:
sub_labels = label_encoder.inverse_transform(pred)

# Submission

In [15]:
submission = pd.read_csv('../data/sample_submission.csv')
submission['class'] = sub_labels

submission.to_csv('../data/submission_stacking_lgbm.csv', index=False)

In [16]:
submission.head()

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


In [17]:
X_train.columns

Index(['lgbm_0', 'lgbm_1', 'lgbm_2', 'cat_0', 'cat_1', 'cat_2', 'xgb_0',
       'xgb_1', 'xgb_2', 'hist_0', 'hist_1', 'hist_2', 'rf_0', 'rf_1', 'rf_2',
       'extra_0', 'extra_1', 'extra_2'],
      dtype='str')